In [2]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from skforecast.datasets import fetch_dataset
from skforecast.recursive import ForecasterRecursive
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.utils import save_forecaster
from skforecast.utils import load_forecaster
from skforecast.preprocessing import DateTimeFeatureTransformer
import joblib
import zipfile
import os

# Download data
# ==============================================================================
data = fetch_dataset(
    name="h2o", raw=True, kwargs_read={"names": ["y", "date"], "header": 0}, verbose=False
)
data['date'] = pd.to_datetime(data['date'], format='%Y-%m-%d')
data = data.set_index('date')
data = data.asfreq('MS')
data

,y
date,
1991-07-01,0.429795
1991-08-01,0.400906
1991-09-01,0.432159
1991-10-01,0.492543
1991-11-01,0.502369
...,...
2008-02-01,0.761822
2008-03-01,0.649435
2008-04-01,0.827887


In [3]:
# Fit export process
# ======================================================================================
calendar_transformer = DateTimeFeatureTransformer(
    features = ['month', 'day_of_month'],
    keep_original_columns = True
)

data = calendar_transformer.fit_transform(X=data)
data_train = data.iloc[:190, :]
data_test = data.iloc[190:, :]


def exponential_weights(index):
    weights = np.exp(-np.arange(len(index)) * 0.01)[::-1]
    return weights / weights.sum() * len(index)

forecaster = ForecasterRecursive(
                 estimator     = RandomForestRegressor(random_state=123, max_depth=3),
                 lags          = 5,
                 forecaster_id = "forecaster_model",
                 transformer_y=StandardScaler(),
                 weight_func=exponential_weights  # custom function
             )

forecaster.fit(y=data_train['y'], exog=data_train.drop(columns='y'))
forecaster.predict(steps=1, exog=data_test.drop(columns='y'))

2007-05-01    0.666284
Freq: MS, Name: pred, dtype: float64

# Pickle

In [5]:
save_forecaster(forecaster, "forecaster.pkl", verbose= True, backend='pickle')

ForecasterRecursive 
Estimator: RandomForestRegressor 
Lags: [1 2 3 4 5] 
Window features: None 
Window size: 5 
Series name: y 
Exogenous included: True 
Exogenous names: month_sin, month_cos, day_of_month_sin, day_of_month_cos 
Categorical features: auto 
Transformer for y: StandardScaler() 
Transformer for exog: None 
Weight function included: True 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('1991-07-01 00:00:00'), Timestamp('2007-04-01 00:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <MonthBegin> 
Estimator parameters: 
    {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth':
    3, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None,
    'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100,
    'n_jobs': None, 'oob_score': False, 'random_state': 123, 'verbose': 0,
    'wa

In [6]:
forecaster = load_forecaster("forecaster.pkl")
forecaster.predict(steps=1, exog=data_test.drop(columns='y'))

ForecasterRecursive 
Estimator: RandomForestRegressor 
Lags: [1 2 3 4 5] 
Window features: None 
Window size: 5 
Series name: y 
Exogenous included: True 
Exogenous names: month_sin, month_cos, day_of_month_sin, day_of_month_cos 
Categorical features: auto 
Transformer for y: StandardScaler() 
Transformer for exog: None 
Weight function included: True 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('1991-07-01 00:00:00'), Timestamp('2007-04-01 00:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <MonthBegin> 
Estimator parameters: 
    {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth':
    3, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None,
    'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100,
    'n_jobs': None, 'oob_score': False, 'random_state': 123, 'verbose': 0,
    'wa

2007-05-01    0.666284
Freq: MS, Name: pred, dtype: float64

# Cloudpickle

In [ ]:
save_forecaster(forecaster, "forecaster.cloudpickle", verbose= False, backend='cloudpickle')

In [ ]:
forecaster = load_forecaster("forecaster.cloudpickle")
forecaster.predict(steps=1, exog=data_test.drop(columns='y'))

# Skops

In [ ]:
forecaster.training_range_

In [ ]:
from skops.io import dump, load, get_untrusted_types

In [ ]:
# Decompose last_window_ and training_range in values and index
last_window_dict = forecaster.last_window_.to_dict(orient="split")
last_window_dict["index"] = [str(ts) for ts in forecaster.last_window_.index]
last_window_dict['freq'] = forecaster.last_window_.index.freqstr
training_range_list = [str(ts) for ts in forecaster.training_range_]
forecaster.last_window_ = last_window_dict
forecaster.training_range_ = training_range_list

# Save object
dump(forecaster, "forecaster.skops")

# Load object
unknown_types = get_untrusted_types(file="forecaster.skops")
forecaster = load(file="forecaster.skops", trusted=unknown_types)

# Recostruct last_window_ and training_range
last_window_ = pd.DataFrame(
    data=forecaster.last_window_["data"], 
    index=pd.to_datetime(forecaster.last_window_["index"]),
    columns=forecaster.last_window_["columns"]
)
last_window_ = last_window_.asfreq(forecaster.last_window_["freq"])

training_range_ = pd.DatetimeIndex(forecaster.training_range_)
forecaster.last_window_ = last_window_
forecaster.training_range_ = training_range_

forecaster.predict(steps=1, exog=data_test)

In [ ]:
# Check what attributes can be serialized with skops
# ==============================================================================
import skops.io

# Assuming 'forecaster' is your trained skforecast object
print("Testing forecaster attributes for skops compatibility...\n")
print("-" * 50)

for attr_name, attr_value in vars(forecaster).items():
    try:
        # Attempt to serialize the individual attribute
        dumps(attr_value)
        print(f"✅ SUCCESS : '{attr_name}' (Type: {type(attr_value).__name__})")
    except Exception as e:
        # If it fails, catch the error and print it
        print(f"❌ FAILED  : '{attr_name}' (Type: {type(attr_value).__name__})")
        print(f"   -> Error: {type(e).__name__}: {e}")

print("-" * 50)
print("\nDiagnostic complete.")